# Machine Learning and Neural Networks
## Problem set 1: Regression

Welcome to tutorial #1. This contains a number of problems for you to solve - some involve solving the equations, and some implementing them in code. You can download this notebook to fill in the gaps.



### Concepts

#### 1:

In a simple linear regression model $y = w_0 + w_1x + ϵ$, what is the role of each term:

* $w_0$: intercept
* $w_1$: slope
* $ϵ$: error

#### 2:

Using the following data:

| x | y |
| --- | --- |
| 1 | 2.8 |
| 2 | 5.1 |
| 3 | 6.7 |

Calculate the optimal values of $w_0$ and $w_1$

* $w_0 = $    w_0 = 0.967
* $w_1 = $    w_1 = 1.95

#### 3:

What happens if we add weights to this data? - Note: This is a hard problem that I only mentioned briefly in Lecture 1. If you feel stuck, move on to the next problem and come back to this one later.

| x | y | $w_y$ |
| --- | --- | -- |
| 1 | 2.8 | 0.4 |
| 2 | 5.1 | 0.3 |
| 3 | 6.7 | 0.6 |

* $w_0 = $    w_0 = 0.246
* $w_1 = $    w_1 =2.375

### Code

#### 4:

Using only pure python and numpy, implement a simple linear regression algorithm. Use it to derive $w_0$ and $w_1$ for the values below.

In [1]:
import numpy as np

x = np.array([4.14, 1.88, 2.11, 7.86, 0.23, 3.44, 8.77, 0.83, 3.74, 7.88, 9.88, 2.55, 2.70, 4.51, 8.14, 5.09, 0.14, 6.87, 9.55, 9.59, ])# 0.1, 2.7,
y = np.array([5.10, 2.04, 1.50, 8.52, 1.65, 3.79, 9.48, 3.26, 5.76, 8.90, 13.45, 3.95, 3.88, 4.97, 8.43, 7.00, 0.66, 7.56, 11.47, 9.20, ])

def simple_linear_regression(x,y):
    x_mean = np.mean(x)
    y_mean = np.mean(y)
    w1 = np.sum((x - x_mean) * (y - y_mean))/np.sum((x - x_mean) ** 2)
    w0 = y_mean - w1 * x_mean
    return w0, w1
w0, w1 = simple_linear_regression(x, y)

print("w0 =", w0)
print("w1 =", w1)
print("Regression equation: y =", w0, "+", w1, "* x")


w0 = 0.8919190705625697
w1 = 1.028344530417904
Regression equation: y = 0.8919190705625697 + 1.028344530417904 * x


#### 5:

The file `inputdata-5.csv` contains data relating sales (gross revenue) at a medium-sized company to advertising spend (all in thousands of GBP per month, adjusted for inflation). Each line corresponds to one month. The left and middle columns in this file show the amount spent on advertising, the right column shows the level of sales (revenue).

Split the data into training and test data, and use linear regression to create models of the form:

$$S=u+k_1 A_1 $$
$$S = u + k_2 A_2 $$
and
$$S = u+ k_1 A_1 + k_2 A_2$$
where $S$ is the revenue, $A_1$ and $A_2$ are the amounts spent on online and TV advertising, respectively.

Then evaluate the each model using the test set. How effective is each model, and what else could you do to improve?

Comment on what is happening in each case, and why any of them perform better or worse than others.



In [3]:
# Enter your solution here:
import numpy as np

# Load the data, skipping the header
data = np.genfromtxt(
    "inputdata-5.csv",
    delimiter=",",
    skip_header=1
)

# Remove any rows containing missing or invalid values
data = data[np.all(np.isfinite(data), axis=1)]

X = data[:, :2]   # OnlineAdvertising, TVAdvertising
y = data[:, 2]    # Revenue

# Split into 80% training and 20% test data
np.random.seed(42)
indices = np.random.permutation(len(y))
split = int(0.8 * len(y))

train_idx = indices[:split]
test_idx = indices[split:]

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


def fit_linear_regression(X, y):
    # Add intercept column
    X_design = np.column_stack((np.ones(len(X)), X))
    return np.linalg.lstsq(X_design, y, rcond=None)[0]


def evaluate_model(X, y, coefficients):
    X_design = np.column_stack((np.ones(len(X)), X))
    predictions = X_design @ coefficients

    rmse = np.sqrt(np.mean((y - predictions) ** 2))
    ss_res = np.sum((y - predictions) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return rmse, r2


# Model 1: Online advertising only
model_online = fit_linear_regression(X_train[:, [0]], y_train)
rmse1, r2_1 = evaluate_model(
    X_test[:, [0]], y_test, model_online
)

# Model 2: Online and TV advertising
model_both = fit_linear_regression(X_train, y_train)
rmse2, r2_2 = evaluate_model(X_test, y_test, model_both)

print("MODEL 1: ONLINE ADVERTISING ONLY")
print(f"Intercept: {model_online[0]:.4f}")
print(f"Online coefficient: {model_online[1]:.4f}")
print(f"Test RMSE: {rmse1:.4f}")
print(f"Test R²: {r2_1:.4f}")

print("\nMODEL 2: ONLINE + TV ADVERTISING")
print(f"Intercept: {model_both[0]:.4f}")
print(f"Online coefficient: {model_both[1]:.4f}")
print(f"TV coefficient: {model_both[2]:.4f}")
print(f"Test RMSE: {rmse2:.4f}")
print(f"Test R²: {r2_2:.4f}")

MODEL 1: ONLINE ADVERTISING ONLY
Intercept: 32.8285
Online coefficient: 6.8381
Test RMSE: 10.3653
Test R²: 0.8216

MODEL 2: ONLINE + TV ADVERTISING
Intercept: 19.4364
Online coefficient: 6.7641
TV coefficient: 2.6048
Test RMSE: 6.6763
Test R²: 0.9260


### Model selection

#### 6.

The file `inputdata-week1.csv` contains a second dataset. The data has 6 columns - 5 independent variables and a dependent variable. Combine the two techniques we discussed in the first lecture, regression and model selection, to determine which independent variables are significant and find the model which best represents the relationship between the independent and dependent variables.

Note: This is a hard problem, and you may not be able to solve it in the time available. Don't be discouraged if that is the case - it is a challenging problem that requires a lot of thought and experimentation.

In [4]:
import numpy as np
from itertools import combinations

# Load data
data = np.genfromtxt(
    "inputdata-week1.csv",
    delimiter=",",
    skip_header=1
)
data = data[np.all(np.isfinite(data), axis=1)]

X = data[:, :5]
y = data[:, 5]

# Train/test split
np.random.seed(42)
indices = np.random.permutation(len(y))
split = int(0.8 * len(y))

train_idx, test_idx = indices[:split], indices[split:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]


def fit_model(X, y):
    A = np.column_stack((np.ones(len(X)), X))
    return np.linalg.lstsq(A, y, rcond=None)[0]


def predict(X, beta):
    A = np.column_stack((np.ones(len(X)), X))
    return A @ beta


def metrics(X, y, beta):
    pred = predict(X, beta)
    rss = np.sum((y - pred) ** 2)
    rmse = np.sqrt(np.mean((y - pred) ** 2))
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - rss / ss_tot if ss_tot > 0 else np.nan
    return rss, rmse, r2


# Evaluate every predictor subset using training BIC
results = []

for size in range(6):
    for cols in combinations(range(5), size):
        cols = list(cols)
        beta = fit_model(X_train[:, cols], y_train)

        rss, _, _ = metrics(X_train[:, cols], y_train, beta)
        _, test_rmse, test_r2 = metrics(
            X_test[:, cols], y_test, beta
        )

        n = len(y_train)
        k = len(cols) + 1  # intercept plus predictors
        bic = n * np.log(max(rss / n, 1e-12)) + k * np.log(n)

        results.append({
            "cols": cols,
            "beta": beta,
            "bic": bic,
            "rmse": test_rmse,
            "r2": test_r2
        })

results.sort(key=lambda r: r["bic"])
best = results[0]

# Fit the full five-variable model for comparison
full_beta = fit_model(X_train, y_train)
_, full_rmse, full_r2 = metrics(X_test, y_test, full_beta)

names = ["x1", "x2", "x3", "x4", "x5"]

print("Number of observations:", len(y))
print("Training observations:", len(train_idx))
print("Test observations:", len(test_idx))

print("\nBEST MODEL BY BIC")
print("Selected variables:", [names[i] for i in best["cols"]])
print("Coefficients:", best["beta"])
print("Training BIC:", round(best["bic"], 4))
print("Test RMSE:", round(best["rmse"], 4))
print("Test R²:", round(best["r2"], 4))

print("\nFULL MODEL")
print("Coefficients:", full_beta)
print("Test RMSE:", round(full_rmse, 4))
print("Test R²:", round(full_r2, 4))

print("\nTOP 5 MODELS")
for r in results[:5]:
    print(
        [names[i] for i in r["cols"]],
        "BIC =", round(r["bic"], 3),
        "RMSE =", round(r["rmse"], 3),
        "R² =", round(r["r2"], 3)
    )
def full_model_statistics(X, y):
    n, p = X.shape
    A = np.column_stack((np.ones(n), X))

    beta = np.linalg.lstsq(A, y, rcond=None)[0]
    residuals = y - A @ beta
    df = n - A.shape[1]

    if df <= 0:
        raise ValueError("Not enough observations.")

    sigma2 = np.sum(residuals**2) / df
    covariance = sigma2 * np.linalg.pinv(A.T @ A)
    se = np.sqrt(np.maximum(np.diag(covariance), 0))
    t_stats = beta / se

    # Normal approximation to two-sided p-values
    # Uses Python's math.erf, no SciPy required
    from math import erf, sqrt
    p_values = np.array([
        1 - erf(abs(t) / sqrt(2)) for t in t_stats
    ])

    return beta, se, t_stats, p_values


beta, se, t_stats, p_values = full_model_statistics(
    X_train, y_train
)

print("Term       Coefficient   Std Error   t statistic   Approx p-value")
print(f"Intercept  {beta[0]:11.4f} {se[0]:11.4f} "
      f"{t_stats[0]:12.3f} {p_values[0]:14.5g}")

for j, name in enumerate(["x1", "x2", "x3", "x4", "x5"], start=1):
    print(f"{name:<10} {beta[j]:11.4f} {se[j]:11.4f} "
          f"{t_stats[j]:12.3f} {p_values[j]:14.5g}")


Number of observations: 100
Training observations: 80
Test observations: 20

BEST MODEL BY BIC
Selected variables: ['x1', 'x2', 'x3', 'x4']
Coefficients: [ 4.04039348  2.21985822  3.57354121  1.51733758 -0.51151405]
Training BIC: 139.5011
Test RMSE: 1.9549
Test R²: 0.9711

FULL MODEL
Coefficients: [ 3.97248023  2.22046531  3.57092595  1.51820734 -0.51129198  0.01497807]
Test RMSE: 1.9413
Test R²: 0.9715

TOP 5 MODELS
['x1', 'x2', 'x3', 'x4'] BIC = 139.501 RMSE = 1.955 R² = 0.971
['x1', 'x2', 'x3', 'x4', 'x5'] BIC = 143.858 RMSE = 1.941 R² = 0.971
['x1', 'x2', 'x3'] BIC = 200.03 RMSE = 2.793 R² = 0.941
['x1', 'x2', 'x3', 'x5'] BIC = 204.329 RMSE = 2.763 R² = 0.942
['x1', 'x3', 'x4'] BIC = 290.382 RMSE = 4.498 R² = 0.847
Term       Coefficient   Std Error   t statistic   Approx p-value
Intercept       3.9725      0.9706        4.093      4.265e-05
x1              2.2205      0.0791       28.064              0
x2              3.5709      0.1710       20.889              0
x3              